# BEDOL core reproducibility notebook

This notebook is a **clean-room reproduction entry point** for the released six-site data. It does not ship cached paper predictions, trained checkpoints, or bootstrap draws. Instead it rebuilds the chronological split, leakage-safe preprocessing, the BEDOL core model, training, held-out metrics, a moving-block bootstrap sanity check, and optional mechanism ablations from the released CSV files.

**Reproducibility boundary.** Exact floating-point values may vary slightly with GPU/CUDA/PyTorch versions. The paper protocol selected architecture and hyperparameters using validation data only and used final seeds 11, 29, and 47. The test year must not be used for model selection.


In [ ]:
# 1. Imports and run mode
from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
CONFIG_DIR = ROOT / "configs"
OUT = ROOT / "reproduction_outputs"
OUT.mkdir(exist_ok=True)

MODE = "FAST"       # change to "PAPER" for the longer 3-seed run
RUN_ABLATIONS = False
RUN_BOOTSTRAP = True

SEEDS = [11] if MODE == "FAST" else [11, 29, 47]
MAX_EPOCHS = 12 if MODE == "FAST" else 120
PATIENCE = 3 if MODE == "FAST" else 15
BOOT_REPS = 500 if MODE == "FAST" else 5000
print("device=", DEVICE, "mode=", MODE, "seeds=", SEEDS)


In [ ]:
# 2. Frozen paper protocol and validation-selected BEDOL parameters
LOOKBACK, H = 168, 24
TRAIN_STRIDE, VAL_STRIDE, TEST_STRIDE = 6, 12, 6
BATCH_SIZE = 128

HP = dict(
    hidden_dim=96, latent_dim=48, site_emb_dim=8,
    dropout=0.11731530334443374,
    lr=7.805065557387398e-4,
    weight_decay=3.585216816340529e-5,
    tf_start=0.670244553854209,
    tf_zero_fraction=0.5973738802854409,
    kl_weight=6.25644792681947e-5,
    prior_sigma=0.5,
)
LOSS_W = dict(huber=0.20, ramp=0.04, energy=0.015, nll=0.08)
WEATHER = [
    "temp_mean","temp_min","temp_max","precip","vapor_pressure","dew_point",
    "sunshine_duration","solar_radiation","cloud_cover","ground_temp",
    "air_temp","wind_speed","air_pressure","relative_humidity"
]
TARGET = "solar_power_filled"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
# 3. Load six site files and reproduce the fixed chronological split
manifest = pd.read_csv(CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")

frames = {}
for _, m in manifest.iterrows():
    p = DATA_DIR / m.filename
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Put the six released *_FILLED.csv files in data/.")
    df = pd.read_csv(p)
    df["ds"] = pd.to_datetime(df["interval_start_kst"])
    df = df.sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    if df[TARGET].isna().any():
        raise ValueError(f"{m.site_id}: {TARGET} has missing values.")
    frames[m.site_id] = df

print("sites:", list(frames))
print("total rows:", sum(len(x) for x in frames.values()))
display(manifest[["site_id","start_date","end_date","rows","target_imputed_rows","quality_flagged_rows"]])


In [ ]:
# 4. Leakage-safe preprocessing
# Weather statistics are estimated from TRAINING data only.
# Future inputs are deterministic calendar terms only; future measured weather is not used.
site_to_idx = {s:i for i,s in enumerate(manifest.site_id.tolist())}
proc, meta = {}, {}

for site, df in frames.items():
    sp = splits.loc[splits.site_id.eq(site)].iloc[0]
    train_end = pd.Timestamp(sp.train_end)
    tr = df[df.ds <= train_end].copy()

    # Stable site scale from training data only.
    cap = float(max(tr[TARGET].max(), 1e-6))
    xcols = [c for c in WEATHER if c in df.columns]
    mu = tr[xcols].median()
    sd = tr[xcols].std().replace(0,1).fillna(1)

    z = df.copy()
    for c in xcols:
        z[c] = z[c].ffill().fillna(mu[c])
        z[c] = (z[c]-mu[c])/sd[c]

    z["y_norm"] = z[TARGET].clip(lower=0)/cap
    hr = z.ds.dt.hour.to_numpy()
    doy = z.ds.dt.dayofyear.to_numpy()
    dow = z.ds.dt.dayofweek.to_numpy()
    z["hour_sin"] = np.sin(2*np.pi*hr/24); z["hour_cos"] = np.cos(2*np.pi*hr/24)
    z["doy_sin"] = np.sin(2*np.pi*(doy-1)/365.25); z["doy_cos"] = np.cos(2*np.pi*(doy-1)/365.25)
    z["dow_sin"] = np.sin(2*np.pi*dow/7); z["dow_cos"] = np.cos(2*np.pi*dow/7)
    z["weekend"] = (dow>=5).astype(float)
    proc[site] = z
    meta[site] = {"capacity":cap}

HIST_FEATURES = ["y_norm"] + [c for c in WEATHER if all(c in proc[s].columns for s in proc)] + [
    "hour_sin","hour_cos","doy_sin","doy_cos","dow_sin","dow_cos","weekend"
]
FUTURE_FEATURES = ["hour_sin","hour_cos","doy_sin","doy_cos","dow_sin","dow_cos","weekend"]
print("historical feature count:", len(HIST_FEATURES))


In [ ]:
# 5. Window index and PyTorch dataset
# A window belongs to train/validation/test according to its forecast origin.
def build_origins(site, split_name, stride):
    df = proc[site]
    sp = splits.loc[splits.site_id.eq(site)].iloc[0]
    if split_name=="train":
        start, end = df.ds.iloc[LOOKBACK-1], pd.Timestamp(sp.train_end)
    elif split_name=="val":
        start, end = pd.Timestamp(sp.val_start), pd.Timestamp(sp.val_end)
    else:
        start, end = pd.Timestamp(sp.test_start), pd.Timestamp(sp.test_end)

    valid = np.where((df.ds.values >= np.datetime64(start)) & (df.ds.values <= np.datetime64(end)))[0]
    valid = valid[(valid >= LOOKBACK-1) & (valid+H < len(df))]
    return valid[::stride]

class SolarWindows(Dataset):
    def __init__(self, split_name):
        stride = {"train":TRAIN_STRIDE,"val":VAL_STRIDE,"test":TEST_STRIDE}[split_name]
        self.items = [(site,int(i)) for site in proc for i in build_origins(site,split_name,stride)]

    def __len__(self): return len(self.items)

    def __getitem__(self, k):
        site,i = self.items[k]
        d = proc[site]
        hist = d.iloc[i-LOOKBACK+1:i+1][HIST_FEATURES].to_numpy(np.float32)
        fut = d.iloc[i+1:i+H+1][FUTURE_FEATURES].to_numpy(np.float32)
        y = d.iloc[i+1:i+H+1]["y_norm"].to_numpy(np.float32)
        return (
            torch.from_numpy(hist), torch.from_numpy(fut),
            torch.tensor(site_to_idx[site]), torch.from_numpy(y),
            site, d.iloc[i+1].ds
        )

train_ds, val_ds, test_ds = SolarWindows("train"), SolarWindows("val"), SolarWindows("test")
print("windows:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# 6. Compact BEDOL implementation
# The implementation follows the paper equations and frozen dimensions.
# It is intentionally auditable rather than a cache of the internal experimental code.
class SelectiveStateEncoder(nn.Module):
    def __init__(self, in_dim, hidden, dropout, fixed_transition=False):
        super().__init__()
        self.in_proj = nn.Sequential(nn.Linear(in_dim,hidden), nn.GELU(), nn.Dropout(dropout))
        self.proposal = nn.Linear(hidden*2, hidden)
        self.gate = nn.Linear(hidden*2, hidden)
        self.fixed_transition = fixed_transition
        self.fixed_alpha = nn.Parameter(torch.zeros(hidden)) if fixed_transition else None

    def forward(self, x):
        h = torch.zeros(x.size(0), self.proposal.out_features, device=x.device)
        alphas = []
        for t in range(x.size(1)):
            r = self.in_proj(x[:,t])
            q = torch.cat([r,h], -1)
            u = torch.tanh(self.proposal(q))
            a = torch.sigmoid(self.fixed_alpha).expand_as(h) if self.fixed_transition else torch.sigmoid(self.gate(q))
            h = a*h + (1-a)*u
            alphas.append(a)
        return h, torch.stack(alphas,1)

class BEDOL(nn.Module):
    def __init__(self, n_hist, n_fut, n_sites, variant="full"):
        super().__init__()
        self.variant = variant
        h, z, se = HP["hidden_dim"], HP["latent_dim"], HP["site_emb_dim"]
        self.encoder = SelectiveStateEncoder(n_hist,h,HP["dropout"],fixed_transition=(variant=="fixed_transition"))
        self.site_emb = nn.Embedding(n_sites,se)
        cond_dim = h if variant=="no_site_embedding" else h+se
        self.mu = nn.Linear(cond_dim,z); self.logvar = nn.Linear(cond_dim,z)
        self.anchor = nn.Linear(LOOKBACK,H)
        dec_in = z + (0 if variant=="no_site_embedding" else se) + n_fut + 2
        self.decoder = nn.GRUCell(dec_in,h)
        self.out = nn.Linear(h,1); self.log_sigma = nn.Linear(h,1); self.res_gate = nn.Linear(h,1)

    def forward(self,hist,fut,site,y_true=None,tf_ratio=0.0,sample=True):
        state, alpha = self.encoder(hist)
        emb = self.site_emb(site)
        cond = state if self.variant=="no_site_embedding" else torch.cat([state,emb],-1)
        mu = self.mu(cond); lv = self.logvar(cond).clamp(-10,6)

        if self.variant=="no_bayesian_latent":
            z = mu
        else:
            eps = torch.randn_like(mu) if (sample and self.training) else torch.zeros_like(mu)
            z = mu + torch.exp(0.5*lv)*eps

        past_y = hist[:,:,0]; last = past_y[:,-1:]
        anchor = self.anchor(past_y-last)+last
        if self.variant=="no_nlinear_anchor": anchor = torch.zeros_like(anchor)

        q, prev, preds, logs = state, last, [], []
        for k in range(H):
            prev_in = prev
            if self.training and y_true is not None and self.variant!="no_teacher_forcing" and random.random()<tf_ratio:
                prev_in = y_true[:,k:k+1]
            pieces = [z]
            if self.variant!="no_site_embedding": pieces.append(emb)
            pieces += [fut[:,k], anchor[:,k:k+1], prev_in]
            q = self.decoder(torch.cat(pieces,-1),q)
            delta = self.out(q); gate = torch.sigmoid(self.res_gate(q))
            pred = (anchor[:,k:k+1] + gate*delta).clamp(min=0)
            preds.append(pred); logs.append(self.log_sigma(q).clamp(-5,3)); prev = pred
        return torch.cat(preds,1), torch.cat(logs,1), mu, lv, alpha


In [ ]:
# 7. Objective and training
def bedol_loss(pred,logs,mu,lv,y):
    err = pred-y
    mse = (err**2).mean()
    hub = torch.nn.functional.huber_loss(pred,y,delta=0.1)
    ramp = ((pred[:,1:]-pred[:,:-1]-(y[:,1:]-y[:,:-1]))**2).mean()
    energy = ((pred.sum(1)-y.sum(1))**2).mean()/H
    nll = (0.5*torch.exp(-2*logs)*err**2 + logs).mean()
    prior_var = HP["prior_sigma"]**2
    kl = 0.5*((torch.exp(lv)+mu**2)/prior_var - 1 - lv + math.log(prior_var)).mean()
    return mse + LOSS_W["huber"]*hub + LOSS_W["ramp"]*ramp + LOSS_W["energy"]*energy + LOSS_W["nll"]*nll + HP["kl_weight"]*kl

def loader(ds,shuffle):
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0,drop_last=False)

@torch.no_grad()
def eval_loss(model, ds):
    model.eval(); vals=[]
    for hist,fut,site,y,_,_ in loader(ds,False):
        hist,fut,site,y = hist.to(DEVICE),fut.to(DEVICE),site.to(DEVICE),y.to(DEVICE)
        p,_,_,_,_ = model(hist,fut,site,sample=False)
        vals.append(float(((p-y)**2).mean().cpu()))
    return np.mean(vals)

def train_one(seed, variant="full"):
    seed_everything(seed)
    model = BEDOL(len(HIST_FEATURES),len(FUTURE_FEATURES),len(site_to_idx),variant).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(),lr=HP["lr"],weight_decay=HP["weight_decay"])
    best, best_state, bad = float("inf"), None, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        frac = epoch/max(1,MAX_EPOCHS-1)
        tf = max(0.0, HP["tf_start"]*(1-frac/HP["tf_zero_fraction"])) if variant!="no_teacher_forcing" else 0.0

        for hist,fut,site,y,_,_ in loader(train_ds,True):
            hist,fut,site,y = hist.to(DEVICE),fut.to(DEVICE),site.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            p,ls,mu,lv,_ = model(hist,fut,site,y_true=y,tf_ratio=tf,sample=True)
            loss = bedol_loss(p,ls,mu,lv,y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()

        v = eval_loss(model,val_ds)
        print(f"seed={seed} variant={variant} epoch={epoch+1:03d} val_mse={v:.6f}")
        if v < best-1e-6:
            best = v
            best_state = {k:x.detach().cpu().clone() for k,x in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE: break

    model.load_state_dict(best_state)
    return model, best


In [ ]:
# 8. Train frozen BEDOL specification
models, val_scores = [], []
for seed in SEEDS:
    m,v = train_one(seed,"full")
    models.append(m); val_scores.append(v)
print("validation MSEs:", val_scores)


In [ ]:
# 9. Held-out predictions and site-macro metrics
@torch.no_grad()
def predict_ensemble(models, ds):
    rows=[]
    for hist,fut,site,y,site_name,origin in loader(ds,False):
        hist,fut,site_t = hist.to(DEVICE),fut.to(DEVICE),site.to(DEVICE)
        ps=[]
        for m in models:
            m.eval()
            p,*_ = m(hist,fut,site_t,sample=False)
            ps.append(p.cpu().numpy())
        pred = np.mean(ps,0); yy = y.numpy()

        for b,s in enumerate(site_name):
            cap = meta[s]["capacity"]
            for h in range(H):
                rows.append((s,pd.Timestamp(origin[b]),h+1,yy[b,h]*cap,pred[b,h]*cap,cap))
    return pd.DataFrame(rows,columns=["site_id","origin","horizon","y_true","pred","capacity"])

def summarize(df):
    rows=[]
    for s,g in df.groupby("site_id"):
        e = g.pred-g.y_true; cap=float(g.capacity.iloc[0])
        rows.append(dict(
            site_id=s, RMSE=np.sqrt(np.mean(e**2)), nRMSE=np.sqrt(np.mean(e**2))/cap,
            MAE=np.mean(np.abs(e)), WAPE=np.abs(e).sum()/np.abs(g.y_true).sum(),
            R2=r2_score(g.y_true,g.pred)
        ))
    site = pd.DataFrame(rows)
    macro = site[["RMSE","nRMSE","MAE","WAPE","R2"]].mean().to_frame().T
    return site,macro

pred = predict_ensemble(models,test_ds)
site_metrics, macro_metrics = summarize(pred)
pred.to_csv(OUT/"bedol_test_predictions.csv",index=False)
site_metrics.to_csv(OUT/"bedol_site_metrics.csv",index=False)
macro_metrics.to_csv(OUT/"bedol_macro_metrics.csv",index=False)
display(site_metrics); display(macro_metrics)
print("Compare with reference/paper_reference_metrics.csv; small numeric differences are expected.")


In [ ]:
# 10. Seasonal-naive sanity check + moving-block bootstrap
naive = pred[["site_id","origin","horizon","y_true","capacity"]].copy()
for s in naive.site_id.unique():
    d = proc[s].set_index("ds")[TARGET]
    mask = naive.site_id.eq(s)
    naive.loc[mask,"pred"] = [
        float(d.get(t+pd.Timedelta(hours=h-25),np.nan))
        for t,h in zip(naive.loc[mask,"origin"],naive.loc[mask,"horizon"])
    ]
naive = naive.dropna(subset=["pred"])

def origin_loss(df):
    x=df.copy()
    x["loss"]=((x.pred-x.y_true)/x.capacity)**2
    return x.groupby(["site_id","origin"],as_index=False).loss.mean()

def block_bootstrap(a,b,block=28,reps=1000,seed=2026):
    d=a.merge(b,on=["site_id","origin"],suffixes=("_a","_b"))
    d["diff"]=d.loss_a-d.loss_b  # <0 means A is better
    rng=np.random.default_rng(seed)
    sites={s:g.sort_values("origin").diff.to_numpy() for s,g in d.groupby("site_id")}
    obs=np.mean([v.mean() for v in sites.values()])
    boots=[]
    for _ in range(reps):
        site_means=[]
        for v in sites.values():
            n=len(v); q=[]
            while len(q)<n:
                st=int(rng.integers(0,n))
                q.extend(v[(np.arange(st,st+block)%n)].tolist())
            site_means.append(np.mean(q[:n]))
        boots.append(np.mean(site_means))
    lo,hi=np.quantile(boots,[.025,.975])
    return obs,lo,hi

if RUN_BOOTSTRAP:
    obs,lo,hi=block_bootstrap(origin_loss(pred),origin_loss(naive),reps=BOOT_REPS)
    print(f"BEDOL - seasonal-naive normalized squared loss: {obs:.6g}; 95% block-bootstrap CI [{lo:.6g}, {hi:.6g}]")


In [ ]:
# 11. Optional mechanism ablations
# Disabled by default because a 3-seed paper run is expensive.
if RUN_ABLATIONS:
    variants=["fixed_transition","no_site_embedding","no_bayesian_latent","no_nlinear_anchor","no_teacher_forcing"]
    records=[]
    for variant in variants:
        ms=[]
        for seed in SEEDS:
            m,_=train_one(seed,variant); ms.append(m)
        q=predict_ensemble(ms,test_ds)
        _,macro=summarize(q)
        records.append({"variant":variant,**macro.iloc[0].to_dict()})
    ablation_summary=pd.DataFrame(records)
    ablation_summary.to_csv(OUT/"ablation_summary.csv",index=False)
    display(ablation_summary)


## Interpretation and honest reproducibility boundary

A successful re-run should reproduce the **qualitative experimental pattern** and yield metrics in the neighborhood of the compact reference CSV. Bit-for-bit equality is not expected across GPUs or software stacks. The full paper analysis used three final seeds, 5,000 moving-block bootstrap replicates, a larger comparator family, and more extensive XAI/ablation diagnostics. This public notebook exposes the essential ingredients needed to independently re-implement and audit the method without distributing cached paper outputs.
